# OGLE data access: from EWS page to normalized observations

This notebook is an executable inspection of the public OGLE-IV Early Warning System adapter. It shows the raw web table, the normalized target table, optional sky/magnitude filters, and the per-event `phot.dat` observations.

In [ ]:
from io import StringIO
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import requests
from target_selection import data_source
from target_selection.sources.ogle import EWS_URL

SEASON = 2026
CACHE_DIR = Path('outputs/cache/ogle_photometry')
OGLE = data_source.ogle
OGLE.describe()

## 1. Inspect the public EWS page

The adapter downloads the HTML page, decodes it as Latin-1, and reads its event table with `pandas.read_html`. This cell shows the raw response and the columns before normalization.

In [ ]:
response = requests.get(EWS_URL, timeout=60)
response.raise_for_status()
raw_html = response.content.decode('latin-1')
raw_tables = pd.read_html(StringIO(raw_html))
raw_events = raw_tables[0]
print(f'HTTP status: {response.status_code}')
print(f'HTML bytes: {len(response.content):,}')
print(f'Tables found: {len(raw_tables)}')
print('Raw columns:')
display(raw_events.head())

## 2. Load normalized OGLE targets

The adapter converts sexagesimal coordinates to degrees and maps `Tmax`, `tau` and `Umin` to `t0_hjd`, `tE_days` and `u0`. The original OGLE columns and event URLs remain available.

In [ ]:
events = OGLE.load_targets(season=SEASON)
print(f'Loaded targets: {len(events):,}')
display(events[['target_id', 'ra_deg', 'dec_deg', 't0_hjd', 'tE_days', 'u0', 'category', 'photometry_url']].head())

## 3. Filter by magnitude and sky region

`magnitude_range` uses the OGLE EWS I-band baseline magnitude (`Ibl`). `ra_range` and `dec_range` use degrees. A right-ascension interval that crosses 0 degrees is supported.

In [ ]:
filtered_events = OGLE.load_targets(
    season=SEASON,
    magnitude_range=(None, 17.0),
    ra_range=(250.0, 280.0),
    dec_range=(-35.0, -20.0),
)
print(f'Filtered targets: {len(filtered_events):,}')
display(filtered_events[['target_id', 'ra_deg', 'dec_deg', 'magnitude_baseline_I']].head())

The same magnitude filter can be expressed through the generic provider filter interface:

In [ ]:
bright_events = OGLE.load_targets(
    season=SEASON,
    filters={'magnitude': {'maximum': 17.0}},
)
print(f'OGLE EWS targets with Ibl <= 17: {len(bright_events):,}')

## 4. Download and inspect one event light curve

Each event page links to a public `phot.dat` file. The adapter caches it locally and returns one normalized observation row per epoch.

In [ ]:
sample = filtered_events.head(1) if len(filtered_events) else events.head(1)
observations = OGLE.photometry(sample, cache_dir=CACHE_DIR)
print(f'Observation rows: {len(observations):,}')
display(observations.head())

if len(observations):
    plt.figure(figsize=(9, 4))
    plt.errorbar(observations['epoch_hjd'], observations['magnitude'], yerr=observations['magnitude_err'], fmt='.', ms=3)
    plt.gca().invert_yaxis()
    plt.xlabel('HJD')
    plt.ylabel('OGLE I magnitude')
    plt.title(f'OGLE EWS photometry: {observations["target_id"].iloc[0]}')
    plt.grid(alpha=0.25)
    plt.show()

## 5. Inspect provenance and cache

The target table keeps the event URL, photometry URL and archive URL. The downloaded text files are stored in the selected cache directory and can be reused in later runs.

In [ ]:
display(sample[['target_id', 'event_url', 'photometry_url', 'archive_url', 'finding_chart_url']])
print(f"Cached files: {len(list(CACHE_DIR.glob('*.phot.dat')))} in {CACHE_DIR}")